In [ ]:
import glob
print(glob.glob("/kaggle/input/**/train_data.jsonl", recursive=True))

In [ ]:
!pip install -q unsloth

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))

import unsloth
print("Unsloth imported OK")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)
print("Qwen2.5-7B loaded in 4-bit!")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,                    # LoRA rank (proven value from notebook)
    lora_alpha = 32,
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # memory-efficient
    random_state = 42,
)
print("LoRA attached!")
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

# load your training file
DATA_PATH = "/kaggle/input/datasets/joesyiem/nemotron-train-data/train_data.jsonl"
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print("Training examples:", len(dataset))

# format each example using the model's chat template
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print("\nSample formatted example:")
print(dataset[0]["text"][:400])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = 2,
        learning_rate = 1e-4,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        logging_steps = 10,
        optim = "adamw_8bit",
        seed = 42,
        output_dir = "/kaggle/working/outputs",
        report_to = "none",
        max_length = 4096,
        dataset_text_field = "text",
    ),
)

print("Trainer ready. Starting training...")
trainer_stats = trainer.train()
print("Training complete!")

In [ ]:
# show the training loss curve
import pandas as pd
history = pd.DataFrame(trainer.state.log_history)
loss_rows = history[history["loss"].notna()][["step", "loss"]]
print("First few losses:")
print(loss_rows.head())
print("\nLast few losses:")
print(loss_rows.tail())
print(f"\nStart loss: {loss_rows['loss'].iloc[0]:.3f}")
print(f"End loss:   {loss_rows['loss'].iloc[-1]:.3f}")

In [ ]:
# save the trained LoRA adapter
model.save_pretrained("/kaggle/working/reasoning_lora")
tokenizer.save_pretrained("/kaggle/working/reasoning_lora")
print("Adapter saved!")

# confirm the files
import os
for f in os.listdir("/kaggle/working/reasoning_lora"):
    size = os.path.getsize(f"/kaggle/working/reasoning_lora/{f}") / 1024
    print(f"  {f}: {size:.1f} KB")

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/reasoning_lora", "zip", "/kaggle/working/reasoning_lora")
print("Zipped! Download reasoning_lora.zip from the Output panel")

import os
print("Zip size:", round(os.path.getsize("/kaggle/working/reasoning_lora.zip")/1024/1024, 1), "MB")

In [ ]:
import glob
print(glob.glob("/kaggle/input/**/train.csv", recursive=True))

In [ ]:
import shutil, sys
shutil.rmtree("/kaggle/working/nemotron-reasoning", ignore_errors=True)
!git clone -q https://github.com/josaiahsyiem/nemotron-reasoning.git
sys.path.insert(0, "/kaggle/working/nemotron-reasoning")
from vcd.solvers.registry import get_solver
from vcd.verify.extract import extract_boxed
from vcd.detect import detect_type
from vcd.solvers.text_encryption import set_vocab
from vcd.vocab import harvest_vocab

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

import pandas as pd
CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(CSV)
df["type"] = df["prompt"].apply(detect_type)
set_vocab(harvest_vocab(CSV))

INSTRUCTION = ("\n\nSolve this step by step. Put your final answer inside "
               "\\boxed{}. For example: \\boxed{your answer}")

def generate(prompt):
    messages = [{"role": "user", "content": prompt + INSTRUCTION}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=1024, temperature=0.3, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# evaluate: 10 held-out puzzles PER TYPE (from the tail, less likely trained on)
results = {}
for ptype in df["type"].unique():
    sub = df[df["type"] == ptype].tail(10)
    correct = 0
    for _, row in sub.iterrows():
        pred = extract_boxed(generate(row["prompt"]))
        if get_solver(ptype).verify(pred, row["answer"]):
            correct += 1
    results[ptype] = correct / len(sub)
    print(f"{ptype}: {correct}/{len(sub)} = {correct/len(sub):.0%}")

overall = sum(results.values()) / len(results)
print(f"\nOverall (avg across types): {overall:.0%}")

In [ ]:
import json
final_results = {
    "numeral_conversion": 1.00, "gravitational_constant": 0.90,
    "text_encryption": 0.80, "unit_conversion": 0.60,
    "bit_manipulation": 0.40, "equation_transformation": 0.00,
    "overall": 0.62,
}
with open("/kaggle/working/eval_results.json", "w") as f:
    json.dump(final_results, f, indent=2)
print("Results saved!")
print("\n=== FINAL: Trained Qwen-7B LoRA, held-out accuracy ===")
for k, v in final_results.items():
    print(f"  {k}: {v:.0%}")